# 03 — FastAPI Endpoints Demo (In-Process ASGI + In-Memory Objects)

Interactive demonstration of all **8 FastAPI endpoints** directly in-process via `httpx.AsyncClient(ASGITransport(app=app))` backed by `InMemoryWorkspace` (no external server or PostgreSQL required).

In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), *Path.cwd().parents, Path.cwd() / "backend"):
    if (candidate / "src" / "main.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from notebooks._common import (
    InMemoryWorkspace,
    banner,
    pretty_print,
    render_material_tree,
    render_misconceptions,
    render_visualization,
)
from tests._harness import (
    sample_chat_payload,
    sample_grade_request,
    sample_material_analyze_request,
)

USE_REAL_LLM = False  # Flip to True to call OpenRouter while keeping DB state in memory

ws = InMemoryWorkspace()
client = await ws.start_api_client(use_real_llm=USE_REAL_LLM)
pretty_print("Seeded In-Memory Context", {
    "class_id": ws.class_id,
    "material_id": ws.material_id,
    "submission_ids": ws.submission_ids,
})

## 1. `GET /api/health`

In [ ]:
health_resp = await client.get("/api/health", headers={"x-request-id": "nb-manual-health"})
pretty_print(
    f"GET /api/health -> {health_resp.status_code} (x-request-id={health_resp.headers.get('x-request-id')})",
    health_resp.json(),
)

## 2. `POST /api/materials/analyze` & `POST /api/materials/tree`

In [ ]:
banner("ENDPOINT 2", "POST /api/materials/analyze & POST /api/materials/tree")
analyze_resp = await client.post(
    "/api/materials/analyze",
    json=sample_material_analyze_request(material_id=ws.material_id, class_id=ws.class_id, long_document=True),
)
pretty_print(f"POST /api/materials/analyze -> {analyze_resp.status_code}", analyze_resp.json())

tree_cached = await client.post(
    "/api/materials/tree",
    json={"material_id": ws.material_id, "force_rebuild": False},
)
print(f"POST /api/materials/tree (cached={tree_cached.json()['cached']}) -> {tree_cached.status_code}")
render_material_tree(tree_cached.json()["tree"])

tree_rebuilt = await client.post(
    "/api/materials/tree",
    json={"material_id": ws.material_id, "force_rebuild": True},
)
pretty_print(f"POST /api/materials/tree (force_rebuild) -> {tree_rebuilt.status_code}", {
    "cached": tree_rebuilt.json()["cached"],
    "total_nodes": tree_rebuilt.json()["tree"]["total_nodes"],
})

## 3. `POST /api/grade`

In [ ]:
grade_results = []
for sub_id in ws.submission_ids:
    resp = await client.post(
        "/api/grade",
        json=sample_grade_request(
            submission_id=sub_id,
            class_id=ws.class_id,
            student_id=ws.student_id,
            material_id=ws.material_id,
        ),
    )
    grade_results.append(resp.json())

pretty_print("POST /api/grade (Submission 1)", grade_results[0])

## 4. `POST /api/consolidation/run`, `GET /api/misconceptions/proposed` & `POST /api/misconceptions/{id}/review`

In [ ]:
import uuid

cons_resp = await client.post(
    "/api/consolidation/run",
    json={"class_id": ws.class_id, "run_type": "daily", "force_dreaming": True},
)
pretty_print(f"POST /api/consolidation/run -> {cons_resp.status_code}", cons_resp.json())

proposed_resp = await client.get("/api/misconceptions/proposed", params={"class_id": ws.class_id})
proposed_items = proposed_resp.json()
print(f"GET /api/misconceptions/proposed -> {len(proposed_items)} items")
render_misconceptions(proposed_items)

if proposed_items:
    misc_id = proposed_items[0]["id"]
    review_resp = await client.post(
        f"/api/misconceptions/{misc_id}/review",
        json={"status": "confirmed", "reviewer_id": str(uuid.uuid4())},
    )
    pretty_print(f"POST /api/misconceptions/{misc_id}/review", review_resp.json())

## 5. `POST /api/chat` (Soft-Warning Gate & Confirmed Reply) + Cleanup

In [ ]:
chat_warn = await client.post(
    "/api/chat",
    json=sample_chat_payload(class_id=ws.class_id, confirmed_soft_warning=False),
)
pretty_print(f"POST /api/chat (Soft Warning) -> {chat_warn.status_code}", chat_warn.json())

chat_ok = await client.post(
    "/api/chat",
    json=sample_chat_payload(class_id=ws.class_id, confirmed_soft_warning=True),
)
chat_ok_json = chat_ok.json()
pretty_print(f"POST /api/chat (Confirmed) -> {chat_ok.status_code}", chat_ok_json)
render_visualization(chat_ok_json.get("visualization"))

pretty_print("Final In-Memory Object Counts", ws.summary())
await ws.stop()